# FraudShield — Phase 1: Exploratory Data Analysis & Leakage Review

This notebook performs the comprehensive data exploration required by Phase 1 of the FraudShield platform:
1. Class Imbalance Quantification (< 1% fraud verification)
2. Amount and Transaction Type Distributions
3. Fraud Vulnerability by Type (`TRANSFER` and `CASH_OUT` focus)
4. Time Structure & Step Sequence Analysis
5. Leakage Review of Post-Transaction Balance Columns
6. Chronological Train / Val / Test Splitting

In [ ]:
from pathlib import Path

import pandas as pd

raw_path = Path("../data/raw/PS_20174392719_1491204439457_log.csv")
df = pd.read_csv(raw_path)
print(f"Loaded {len(df):,} transactions.")
df.head()

## 1. Class Balance & Imbalance Quantification

In [ ]:
fraud_counts = df["isFraud"].value_counts()
fraud_rate = df["isFraud"].mean() * 100
print("Class Balance:")
print(fraud_counts)
print(f"Fraud Rate: {fraud_rate:.4f}% (< 1% requirement met)")

## 2. Fraud Rate by Transaction Type

In [ ]:
type_summary = df.groupby("type")["isFraud"].agg(["count", "sum", "mean"])
type_summary["fraud_pct"] = type_summary["mean"] * 100
print(type_summary)
# In PaySim, fraud only occurs in TRANSFER and CASH_OUT!

## 3. Amount Distribution Comparison

In [ ]:
print("--- Legitimate Amount Statistics ---")
print(df[df["isFraud"] == 0]["amount"].describe())
print("\n--- Fraudulent Amount Statistics ---")
print(df[df["isFraud"] == 1]["amount"].describe())

## 4. Balance Leakage Review

In [ ]:
# How often does fraud drain the account to zero?
fraud_drain_rate = (df[df["isFraud"] == 1]["newbalanceOrig"] == 0).mean() * 100
legit_drain_rate = (df[df["isFraud"] == 0]["newbalanceOrig"] == 0).mean() * 100
print(f"Fraud cases with newbalanceOrig == 0: {fraud_drain_rate:.2f}%")
print(f"Legitimate cases with newbalanceOrig == 0: {legit_drain_rate:.2f}%")
# newbalanceOrig and newbalanceDest are post-transaction state and MUST be dropped to prevent target leakage.

## 5. Time Structure & Time-Based Splits

In [ ]:
print(f"Step range: {df['step'].min()} to {df['step'].max()} (hours)")
train_split = df[df["step"] <= 323]
val_split = df[(df["step"] > 323) & (df["step"] <= 378)]
test_split = df[df["step"] > 378]
print(
    f"Train: {len(train_split):,} rows, Fraud rate: {train_split['isFraud'].mean() * 100:.4f}%"
)
print(
    f"Val:   {len(val_split):,} rows, Fraud rate: {val_split['isFraud'].mean() * 100:.4f}%"
)
print(
    f"Test:  {len(test_split):,} rows, Fraud rate: {test_split['isFraud'].mean() * 100:.4f}%"
)